In [1]:
import sys
sys.path.append("..")
import pandas as pd
from helper_functions.prep_data_function import prepare_train_test
X_train, X_test, y_train, y_test = prepare_train_test("../data/cab_rides_features.csv")
X_train

,cab_type,name,source,destination,time_of_day,demand_density,distance,rides_in_hour_at_source,bad_weather_score,is_raining,is_weekend,ride_hour,uber_lyft_price_ratio
0,Lyft,Lux Black XL,Boston University,Theatre District,late_night,low,3.03,9,0.311602,False,False,3,NaN
1,Uber,Black,South Station,Theatre District,late_night,low,1.30,11,0.319336,False,False,3,NaN
2,Lyft,Lyft,Northeastern University,Beacon Hill,late_night,low,2.43,4,0.321602,False,False,3,NaN
3,Uber,UberXL,Theatre District,Fenway,late_night,low,2.71,13,0.318420,False,False,3,2.755102
4,Uber,UberX,Theatre District,Fenway,late_night,low,2.71,13,0.318420,False,False,3,2.755102
...,...,...,...,...,...,...,...,...,...,...,...,...,...
510380,Lyft,Lux,South Station,West End,evening,low,2.00,123,0.360612,False,True,17,0.802513
510379,Lyft,Lyft XL,South Station,West End,evening,low,2.00,123,0.360612,False,True,17,0.802513
510376,Lyft,Lux Black,South Station,West End,evening,low,2.00,123,0.360612,False,True,17,0.802513
510377,Lyft,Lyft,South Station,West End,evening,low,2.00,123,0.360612,False,True,17,0.802513


In [2]:
baseline_lookup = y_train.groupby(X_train["demand_density"]).median()
baseline_lookup

demand_density
high         1.117647
low          1.119185
medium       1.118644
very_high    1.119565
Name: effective_surge, dtype: float64

In [3]:
baseline_pred = X_test["demand_density"].map(baseline_lookup).astype(float)
baseline_pred.head()

510386    1.117647
510387    1.117647
510384    1.117647
510383    1.117647
510382    1.117647
Name: demand_density, dtype: float64

In [4]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
baseline_r2 = r2_score(y_true=y_test,y_pred=baseline_pred)
baseline_rmse = mean_squared_error(y_true=y_test,y_pred=baseline_pred)**0.5
baseline_mae = mean_absolute_error(y_true=y_test,y_pred=baseline_pred)
baseline_r2, baseline_rmse, baseline_mae

(-0.08999355980939416, 0.31443456103033834, 0.1777986061100099)

In [5]:
full_df = pd.read_csv('../data/cab_rides_features.csv',parse_dates=["ride_time"])
test_actuals = full_df.loc[X_test.index]
test_actuals[["price","distance","base_price_per_mile"]].head()

,price,distance,base_price_per_mile
510386,9.0,4.31,0.957123
510387,7.0,1.37,5.035971
510384,3.0,1.37,2.222222
510383,7.0,1.01,6.603774
510382,10.5,1.37,7.720588


In [6]:
actual_revenue = test_actuals["price"].sum()
baseline_predicted_revenue = (test_actuals["base_price_per_mile"] * test_actuals["distance"] * baseline_pred).sum()
actual_revenue, baseline_predicted_revenue

(np.float64(2112140.35), np.float64(2042304.3261624188))

In [7]:
baseline_revenue_diff_pct = (baseline_predicted_revenue - actual_revenue) / actual_revenue * 100
baseline_revenue_diff_pct

np.float64(-3.306410193696706)